<a href="https://colab.research.google.com/github/arshdeepbangar/AAI2025/blob/main/Agentic_AI__in_Customer_Service_and_Sales_Chatbot_Gemini_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic AI in Customer Service and Sales Chatbot


## 1. Install Libraries


In [11]:
%pip -q install -U langgraph langchain-core langchain-google-genai


## 2. Gemini API Key


In [12]:
import os
from google.colab import userdata

try:
    api_key = userdata.get("GOOGLE_API_KEY")
    if not api_key or not api_key.strip():
        raise ValueError("GOOGLE_API_KEY is empty.")
    os.environ["GOOGLE_API_KEY"] = api_key.strip()
    print("Gemini API key loaded.")
except Exception:
    os.environ.pop("GOOGLE_API_KEY", None)
    print("Add GOOGLE_API_KEY to Colab Secrets and enable notebook access.")

os.environ["GEMINI_MODEL"] = "gemini-3.5-flash-lite"


Gemini API key loaded.


## 3. Chatbot and Agents


In [13]:
import os
import re
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph, add_messages


# Sample store records
ORDERS = {
    "1001": {"item": "Wireless Headphones", "status": "Shipped", "arrival": "October 26, 2026"},
    "1002": {"item": "Running Shoes", "status": "Processing", "arrival": "October 29, 2026"},
    "1003": {"item": "Laptop Backpack", "status": "Delivered", "arrival": "October 20, 2026"},
}
PRODUCTS = [
    {"name": "Wireless Headphones", "category": "audio", "price": 79, "detail": "wireless, lightweight"},
    {"name": "Bluetooth Speaker", "category": "audio", "price": 49, "detail": "portable, water-resistant"},
    {"name": "Running Shoes", "category": "fitness", "price": 89, "detail": "cushioned, everyday running"},
    {"name": "Yoga Mat", "category": "fitness", "price": 25, "detail": "non-slip, easy to carry"},
    {"name": "Laptop Backpack", "category": "bags", "price": 59, "detail": "padded laptop sleeve"},
    {"name": "Travel Duffel", "category": "bags", "price": 45, "detail": "compact travel bag"},
]
RETURN_POLICY = (
    "Demo store policy: unused items can be returned within 30 days of delivery "
    "with proof of purchase. Refunds go to the original payment method, usually "
    "within 5–10 business days after approval. This is a fictional classroom policy."
)
ROLE_PROMPT = (
    "You are ShopSmart's friendly retail customer service assistant. "
    "Be clear, concise, and helpful. You are operating on DEMO records only. "
    "Do not invent order information, inventory, delivery guarantees, refund approvals, "
    "or actions taken. The supplied facts are authoritative. "
    "Remember relevant prior messages, and politely ask follow-up questions when needed."
)


class SupportState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    route: str
    last_route: str
    order_id: str


def last_customer_text(state: SupportState) -> str:
    return next((str(m.content) for m in reversed(state.get("messages", []))
                 if isinstance(m, HumanMessage)), "")


def choose_route(state: SupportState) -> str:
    query = last_customer_text(state).lower()
    refund_words = ("refund", "return", "exchange", "send back", "money back")
    order_words = ("order", "delivery", "shipped", "shipping", "tracking", "package", "arrive", "status")
    product_words = ("recommend", "suggest", "buy", "product", "looking for", "under $", "headphones", "shoes", "backpack", "speaker", "yoga", "duffel")
    if any(w in query for w in refund_words):
        return "refund"
    if any(w in query for w in order_words) or re.search(r"\b100[123]\b", query):
        return "orders"
    if any(w in query for w in product_words):
        return "products"
    return state.get("last_route", "products")


def router(state: SupportState) -> dict:
    return {"route": choose_route(state)}


def extract_order_id(state: SupportState) -> str:
    match = re.search(r"\b(\d{4})\b", last_customer_text(state))
    return match.group(1) if match else state.get("order_id", "")


def generate_response(state: SupportState, agent_role: str, facts: str, offline: bool, fallback: str) -> str:
    if offline:
        return fallback
    from langchain_google_genai import ChatGoogleGenerativeAI
    model = ChatGoogleGenerativeAI(model=os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite"), temperature=0, max_retries=2)
    messages = [SystemMessage(content=ROLE_PROMPT + "\nYour specialty: " + agent_role +
                              "\nUse ONLY these facts for policy, products, and orders:\n" + facts)]
    messages.extend(state.get("messages", [])[-12:])
    response = model.invoke(messages)
    return str(response.content)


def build_graph(offline: bool = False):
    if not offline and not os.getenv("GOOGLE_API_KEY"):
        raise RuntimeError("GOOGLE_API_KEY is missing. Add it to Colab Secrets.")

    def order_agent(state: SupportState) -> dict:
        order_id = extract_order_id(state)
        order = ORDERS.get(order_id)
        if order:
            facts = f"Order {order_id}: {order['item']}; status: {order['status']}; estimated/delivered date: {order['arrival']}."
            fallback = f"Order #{order_id} ({order['item']}) is {order['status'].lower()}. Date listed: {order['arrival']}."
        else:
            facts = "No matching order is available. Ask for a sample order ID: 1001, 1002, or 1003."
            fallback = "Please provide a demo order number (1001, 1002, or 1003) so I can check its status."
        answer = generate_response(state, "Order status agent", facts, offline, fallback)
        return {"messages": [AIMessage(content=answer)], "last_route": "orders", "order_id": order_id}

    def refund_agent(state: SupportState) -> dict:
        answer = generate_response(state, "Refund policy agent", RETURN_POLICY, offline, RETURN_POLICY)
        return {"messages": [AIMessage(content=answer)], "last_route": "refund"}

    def product_agent(state: SupportState) -> dict:
        query = last_customer_text(state).lower()
        category = next((c for c in ("audio", "fitness", "bags") if c in query), None)
        if not category:
            category = "audio" if any(k in query for k in ("headphone", "speaker", "music")) else (
                "fitness" if any(k in query for k in ("run", "shoe", "yoga", "exercise")) else (
                    "bags" if any(k in query for k in ("bag", "backpack", "travel")) else None))
        ceiling = re.search(r"(?:under|below|less than)\s*\$?(\d+)", query)
        budget = int(ceiling.group(1)) if ceiling else None
        selected = [p for p in PRODUCTS if (not category or p["category"] == category)
                    and (budget is None or p["price"] < budget)]
        facts = "Available sample catalog: " + "; ".join(
            f"{p['name']} (${p['price']}, {p['detail']})" for p in selected)
        if not selected:
            facts = "No matching demo products. Ask for another category or budget."
        fallback = ("I recommend " + "; ".join(f"{p['name']} (${p['price']})" for p in selected[:3]) + ".") if selected else (
            "I could not find a matching demo product. Would you like to try a different budget or category?")
        answer = generate_response(state, "Product recommendation agent", facts, offline, fallback)
        return {"messages": [AIMessage(content=answer)], "last_route": "products"}

    workflow = StateGraph(SupportState)
    workflow.add_node("router", router)
    workflow.add_node("orders", order_agent)
    workflow.add_node("refund", refund_agent)
    workflow.add_node("products", product_agent)
    workflow.add_edge(START, "router")
    workflow.add_conditional_edges("router", lambda state: state["route"],
                                   {"orders": "orders", "refund": "refund", "products": "products"})
    for node in ("orders", "refund", "products"):
        workflow.add_edge(node, END)
    return workflow.compile(checkpointer=MemorySaver())


def ask(graph, question: str, thread_id: str = "class-demo") -> tuple[str, str]:
    result = graph.invoke({"messages": [HumanMessage(content=question)]},
                          config={"configurable": {"thread_id": thread_id}})
    return result["messages"][-1].content, result["last_route"]


def run_demo(graph):
    questions = [
        "Where is order 1001?",
        "When will that order arrive?",
        "What is your refund policy?",
        "How long does that refund take?",
        "Recommend headphones under $100.",
    ]
    print("Offline sample conversation\n")
    for index, q in enumerate(questions, 1):
        answer, route = ask(graph, q)
        print(f"TURN {index}  [agent: {route}]\nCustomer: {q}\nAssistant: {answer}\n")
    stored = graph.get_state({"configurable": {"thread_id": "class-demo"}}).values
    print(f"MemorySaver check: {len(stored['messages'])} messages saved (5 user + 5 assistant).")



## 4. Offline Demo


In [14]:
offline_graph = build_graph(offline=True)
run_demo(offline_graph)

Offline sample conversation

TURN 1  [agent: orders]
Customer: Where is order 1001?
Assistant: Order #1001 (Wireless Headphones) is shipped. Date listed: October 26, 2026.

TURN 2  [agent: orders]
Customer: When will that order arrive?
Assistant: Order #1001 (Wireless Headphones) is shipped. Date listed: October 26, 2026.

TURN 3  [agent: refund]
Customer: What is your refund policy?
Assistant: Demo store policy: unused items can be returned within 30 days of delivery with proof of purchase. Refunds go to the original payment method, usually within 5–10 business days after approval. This is a fictional classroom policy.

TURN 4  [agent: refund]
Customer: How long does that refund take?
Assistant: Demo store policy: unused items can be returned within 30 days of delivery with proof of purchase. Refunds go to the original payment method, usually within 5–10 business days after approval. This is a fictional classroom policy.

TURN 5  [agent: products]
Customer: Recommend headphones under 

## 5. Conversation Memory


In [15]:
state = offline_graph.get_state({"configurable": {"thread_id": "class-demo"}}).values
print("Messages saved:", len(state["messages"]))
print("Most recent agent:", state.get("last_route"))
print("Order ID:", state.get("order_id"))
assert len(state["messages"]) >= 6

Messages saved: 10
Most recent agent: products
Order ID: 1001


## 6. Gemini Demo


In [16]:
if os.getenv("GOOGLE_API_KEY"):
    live_graph = build_graph(offline=False)
    questions = [
        "Where is order 1001?",
        "When will that order arrive?",
        "What is your refund policy?",
        "How long does that refund take?",
        "Recommend headphones under $100."
    ]
    for number, question in enumerate(questions, 1):
        response, agent = ask(live_graph, question, thread_id="live-demo")
        print(f"Turn {number} [{agent}]\nCustomer: {question}\nAssistant: {response}\n")
    state = live_graph.get_state({"configurable": {"thread_id": "live-demo"}}).values
    print("Messages saved:", len(state["messages"]))
else:
    print("Add GOOGLE_API_KEY in Colab Secrets to run the Gemini demo.")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 1 [orders]
Customer: Where is order 1001?
Assistant: [{'type': 'text', 'text': "Hello! I'd be happy to help you with order 1001. \n\nAccording to our demo records, order 1001 (Wireless Headphones) has been **Shipped** and has an estimated or delivered date of **October 26, 2026**. \n\nIs there anything else I can help you with today?", 'extras': {'signature': 'EmAKXgFpFH0TnEb71XF9R5YOmg3dXuyBBywNtsLu9GTvibyqNw2/9pK0UlDeHI2Mi5ZEdsQGuIhQBxEDwcXacAvTx9Rm7ojGCWbUm87tGe09oEsYdJY5H9pcz0rQupXmEHI='}}]



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 2 [orders]
Customer: When will that order arrive?
Assistant: [{'type': 'text', 'text': 'According to our demo records, the estimated or delivered date for order 1001 is **October 26, 2026**. \n\nLet me know if you need help with anything else!', 'extras': {'signature': 'EmAKXgFpFH0TdBIFSF7/9jRCDkkgc4oCA/qw9dpSeiD2y/Mn7QkJXyVVUtVccgrvolRW4F9Li9knHWztHeQ7WnCs4YK+k5iZZCgdKR7KSq1wSyriZGP/Pag2BsdKZH6BQEQ='}}]



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 3 [refund]
Customer: What is your refund policy?
Assistant: [{'type': 'text', 'text': "Hello! I'd be happy to share our refund policy. \n\nAccording to our demo records, unused items can be returned within 30 days of delivery with proof of purchase. Refunds are issued to the original payment method, which usually takes 5–10 business days after approval. \n\nPlease note that this is a fictional classroom policy. Let me know if you have any other questions!", 'extras': {'signature': 'EmAKXgFpFH0TUFYpwIbOkZh2Q8qGxq7PE+OZepUNLor4prjGf7EHoYXOZJIiLBmXb328KIGYqR29UPjQj6kmJe3bPKABYWKklDj3hh0lggLuWAxflGYYtCvwGU+j4bH2fdk='}}]



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 4 [refund]
Customer: How long does that refund take?
Assistant: [{'type': 'text', 'text': 'Based on our demo records, refunds are issued to the original payment method and usually take 5–10 business days after approval. \n\nPlease let me know if you have any other questions!', 'extras': {'signature': 'EmAKXgFpFH0TKVdI+CpzpSuhZ99JxLF5hM9cwhh/J9j6iNTZMTfTLlTaEabSjTkL1dn9WuUlk0ZB+lchZBE7Suhic3V0UFFO/UAuzgxFaQY/gQgkrVzj12l8cmbGrnWjr5U='}}]



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 5 [products]
Customer: Recommend headphones under $100.
Assistant: [{'type': 'text', 'text': "Hello! I'd be happy to help with that. \n\nFrom our sample catalog, I recommend our **Wireless Headphones** for $79. They are wireless and lightweight! \n\nWould you like to know more about them, or are you interested in our Bluetooth Speaker as well?", 'extras': {'signature': 'EmAKXgFpFH0TjhdqOajjeujlBjArKAXNjGXGLGd6/8mqVGFz+LOmThZniwwMznTWtyk6Cfgvj4wb8kqSutFZ2Zi3sESiUwAc7NYOs5UWXEWncVBMHN7KV0AnCgU4tZcdDU4='}}]

Messages saved: 10


## 7. Ask a Question


In [17]:
question = "Can you check order 1002?"
thread_id = "my-customer-session"

if os.getenv("GOOGLE_API_KEY"):
    if "customer_graph" not in globals():
        customer_graph = build_graph(offline=False)
    response, agent = ask(customer_graph, question, thread_id=thread_id)
    print(f"Agent: {agent}\nCustomer: {question}\nAssistant: {response}")
else:
    print("Add GOOGLE_API_KEY in Colab Secrets to use live chat.")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Agent: orders
Customer: Can you check order 1002?
Assistant: [{'type': 'text', 'text': "Hello! I'd be happy to check that for you. \n\nAccording to our demo records, Order 1002 (Running Shoes) is currently **Processing**, with an estimated delivery date of **October 29, 2026**. \n\nLet me know if you need help with anything else!", 'extras': {'signature': 'EmAKXgFpFH0TW7ke2ck2ScOSvCwOBmyQYZL0QzG1qworqJpuHXa3U+GF7ht3gtuGMtb2Vz574wW1hGtuXsNUd1yl+oEbtS9ydldWYyrCj0Z247SiymLROwJu3e3EInh/ewQ='}}]
